# Chapter 65: Recommendation Systems

How does a capped union of candidate sources change recall, and what is the correct metric-specific ranking ceiling?

The chapter gives me a usable system shape: retrieve plausible items, then build a familiar ranker on realistic candidates. The cold-start routing discussion changes what I would implement. I would reread the recall-ceiling formula because its aggregation differs from MAP. The companion keeps retrieval fixed and derives a ranking bound under one explicit metric rather than transferring recall numbers blindly. For three constructed users, union ordered history, co-visitation, and popularity lists while deduplicating. Keep the selected candidate cap. Compute micro and macro recall, and the oracle AP@3 if every retrieved relevant item were ranked first. Return per-user truth counts, hits, recall, and oracle AP. Micro recall and macro AP need not share a numeric ceiling because their denominators differ. Candidate coverage constrains ranking, but the bound must respect the metric’s units. A larger union before truncation can still lose useful items if source priority pushes them outside the cap. Compare candidate caps one, four, and eight. Identify which source contributes each captured item, then explain why a correct item lost during truncation cannot be recovered by a reranker. Trace the denominators before using a candidate ceiling in a modeling decision.

This is a constructed teaching example. Its results describe these supplied inputs, not a measured competition gain.

## Assumptions

Binaryrelevanceand AP@3 denominatormin(numberoftrueitems, 3). Oracle is an analytical upperboundfor this exact candidate set, not a trainedranker. Sourceorderdeterminestruncationpriority.

In [1]:
parameter = 4
# Change this input, then run the following calculation.

In [2]:
import json
assert int(parameter)==parameter and 1<=parameter<=8
cap=int(parameter); truth=[{2},{4,5,6,7},{8,9}]
sources=[[[1,2],[3,4],[5,6]],[[4,1],[5,6],[7,8]],[[1,8],[9,2],[3,4]]]
rows=[]; totalhits=tottruth=0
for user,(ys,lists) in enumerate(zip(truth,sources)):
 union=list(dict.fromkeys(item for items in lists for item in items)); chosen=union[:cap]; hits=len(set(chosen)&ys)
 oracle=min(hits,3)/min(len(ys),3)
 rows.append({'user':user,'truth_items':len(ys),'hits':hits,'candidates':chosen,'recall':hits/len(ys),'oracle_ap_at_3':oracle})
 totalhits+=hits; tottruth+=len(ys)
assert all(0<=r['oracle_ap_at_3']<=1 for r in rows)
print(json.dumps({'cap':cap,'micro_recall':totalhits/tottruth,'macro_recall':sum(r['recall'] for r in rows)/3,'oracle_map_at_3':sum(r['oracle_ap_at_3'] for r in rows)/3,'table':rows}))


{"cap": 4, "micro_recall": 0.8571428571428571, "macro_recall": 0.9166666666666666, "oracle_map_at_3": 1.0, "table": [{"user": 0, "truth_items": 1, "hits": 1, "candidates": [1, 2, 3, 4], "recall": 1.0, "oracle_ap_at_3": 1.0}, {"user": 1, "truth_items": 4, "hits": 3, "candidates": [4, 1, 5, 6], "recall": 0.75, "oracle_ap_at_3": 1.0}, {"user": 2, "truth_items": 2, "hits": 2, "candidates": [1, 8, 9, 2], "recall": 1.0, "oracle_ap_at_3": 1.0}]}


## Interpretation

Micro recall and macro APneednot share a numeric ceiling because their denominators differ. Candidate coverage constrains ranking, the bound must respect metric units. A larger union before truncation can still lose useful items if source priority pushes them outside the cap.

Book location: Chapter 65, ## Candidate Generation: Union Several Weak Sources.

[Technical corrections](https://github.com/KarpelesPublishing/winning-kaggle-companion/blob/main/docs/errata.md).